# Situación 3 — Análisis Complementario
## Moran I + LISA · LOO-CV por Estación · Semivariogramas Obs vs Pred · Ablación DL vs DL+Kriging

Notebook complementario que cubre todos los entregables ausentes en `convlstm_geovision.ipynb` y `kriging_era5_ked.ipynb`:

| Análisis | Requis. PDF | Estado anterior | Este notebook |
|---|---|---|---|
| Moran I Global (9 escenarios) | `I > 0.30, p < 0.05` | ❌ Ausente | ✅ |
| LISA + mapas de significancia | Clusters HH/LL | ❌ Ausente | ✅ |
| LOO-CV tabla por estación | RMSE, R², IC95% | ❌ Output borrado | ✅ |
| Comparación semivariogramas obs vs pred | Coherencia espacial | ❌ Ausente | ✅ |
| Ablación DL solo vs DL+Kriging | Evidencia cuantitativa | ❌ Ausente | ✅ |
| KPI Dashboard final | Todos los umbrales | ❌ Parcial | ✅ |

**Datos reales**: se leen desde `gold.sat` (Wasabi S3) — sin sintéticos.

## 0. Dependencias e Importaciones

In [ ]:
# ── Instalación (una sola vez) ───────────────────────────────────────────────
# !pip install -q esda libpysal pykrige s3fs zarr boto3 scikit-learn skgstat

import os, io, warnings, hashlib, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from pathlib import Path
from itertools import combinations

# Deep Learning
import torch
import torch.nn as nn
import torch.nn.functional as F

# Geoestadística
from pykrige.ok import OrdinaryKriging
from pykrige.ok3d import OrdinaryKriging3D
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Moran I + LISA
import esda
from esda import Moran, Moran_Local
import libpysal
from libpysal.weights import lat2W, KNN

# Wasabi / Cloud
import boto3
import s3fs
import zarr
from scipy.interpolate import RegularGridInterpolator
import skgstat as skg

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PROJECT_ROOT = Path('/teamspace/studios/this_studio/GeoVision-CLIP-Cali')
MODELS_DIR   = PROJECT_ROOT / 'models'
FIGS_DIR     = PROJECT_ROOT / 'figuras' / 'sit3_complemento'
FIGS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device  : {DEVICE}")
print(f"PyTorch : {torch.__version__}")
print(f"esda    : {esda.__version__}")
print(f"Figuras : {FIGS_DIR}")

## 1. Credenciales Wasabi + Constantes

In [ ]:
# ── Credenciales Wasabi (igual que notebooks Situación 1) ────────────────────
WASABI_ACCESS_KEY = os.environ.get("WASABI_ACCESS_KEY", "EPAJLZS1BT5K3X8CPGR2")
WASABI_SECRET_KEY = os.environ.get("WASABI_SECRET_KEY", "QnJ9u6PCBlKUDUpGMMsjCAkOTaRaTYJY6uwl8hek")
ENDPOINT_URL      = "https://s3.us-west-1.wasabisys.com"
REGION            = "us-east-1"
BUCKET_GOLD       = "gold.sat"
BUCKET_SILVER     = "silver.sat"

s3 = boto3.client("s3",
    endpoint_url=ENDPOINT_URL,
    aws_access_key_id=WASABI_ACCESS_KEY,
    aws_secret_access_key=WASABI_SECRET_KEY,
    region_name=REGION)

s3_fs = s3fs.S3FileSystem(
    key=WASABI_ACCESS_KEY, secret=WASABI_SECRET_KEY,
    endpoint_url=ENDPOINT_URL,
    client_kwargs={"region_name": REGION})

try:
    s3.list_objects_v2(Bucket=BUCKET_GOLD, MaxKeys=1)
    print("[OK] Wasabi gold.sat conectado")
except Exception as e:
    print(f"[ERROR] {e}")

# ── Constantes del proyecto ───────────────────────────────────────────────────
BBOX = {"lat_min": 3.30, "lat_max": 3.55, "lon_min": -76.60, "lon_max": -76.40}
G    = 8        # grid ConvLSTM
CONTAMINANTS = ["NO2", "SO2", "O3"]
HORIZONS     = ["T1",  "T3",  "T7"]
HORIZON_DAYS = [1, 3, 7]
FINE_H, FINE_W = 40, 50   # grilla fina KED
CONT_UNITS   = "μg/m³"

# Umbrales KPI del PDF
KPI_RMSE_MIN  = {"NO2": 8,  "SO2": 6,  "O3": 12}
KPI_RMSE_EXC  = {"NO2": 4,  "SO2": 3,  "O3": 6}
KPI_R2_MIN    = 0.55
KPI_MORAN_MIN = 0.30
KPI_COV95_MIN = 0.92

# Estaciones DAGMA del PDF
DAGMA_STATIONS = {
    "Univalle"    : {"lat": 3.376, "lon": -76.536},
    "Lili"        : {"lat": 3.360, "lon": -76.548},
    "Pance"       : {"lat": 3.329, "lon": -76.565},
    "Cañaveralejo": {"lat": 3.418, "lon": -76.538},
    "SENA"        : {"lat": 3.453, "lon": -76.503},
    "Pasoancho"   : {"lat": 3.392, "lon": -76.545},
    "Compartir"   : {"lat": 3.405, "lon": -76.516},
    "Industriales": {"lat": 3.464, "lon": -76.507},
    "Guaduales"   : {"lat": 3.368, "lon": -76.571},
}

def coord_to_grid(lat, lon):
    r = int((BBOX["lat_max"] - lat) / (BBOX["lat_max"] - BBOX["lat_min"]) * G)
    c = int((lon - BBOX["lon_min"]) / (BBOX["lon_max"] - BBOX["lon_min"]) * G)
    return min(max(r, 0), G-1), min(max(c, 0), G-1)

for sname, sinfo in DAGMA_STATIONS.items():
    sinfo["gr"], sinfo["gc"] = coord_to_grid(sinfo["lat"], sinfo["lon"])

CELL_LATS = np.array([BBOX["lat_max"] - (r + .5) * (BBOX["lat_max"]-BBOX["lat_min"])/G for r in range(G)])
CELL_LONS = np.array([BBOX["lon_min"] + (c + .5) * (BBOX["lon_max"]-BBOX["lon_min"])/G for c in range(G)])

print("Constantes cargadas:", G, "×", G, "grid,", len(DAGMA_STATIONS), "estaciones DAGMA")

## 2. Carga de Datos Reales desde Wasabi (gold.sat)

In [ ]:
def read_parquet_gold(key):
    obj = s3.get_object(Bucket=BUCKET_GOLD, Key=key)
    return pd.read_parquet(io.BytesIO(obj["Body"].read()))

print("[1/3] Cargando ground_truth/consolidado.parquet ...")
df_gt = read_parquet_gold("ground_truth/consolidado.parquet")
df_gt["fecha"] = pd.to_datetime(df_gt["fecha"])
print(f"  → {len(df_gt):,} filas | contaminantes: {sorted(df_gt['contaminante'].unique())}")

print("[2/3] Cargando ground_truth/estaciones_coords.parquet ...")
df_coords = read_parquet_gold("ground_truth/estaciones_coords.parquet")
print(f"  → {len(df_coords)} estaciones")
print(df_coords[["estacion","lat","lon"]].to_string(index=False))

print("[3/3] Cargando ground_truth/loocv_all.parquet ...")
df_loocv_raw = read_parquet_gold("ground_truth/loocv_all.parquet")
df_loocv_raw["fecha"] = pd.to_datetime(df_loocv_raw["fecha"])
print(f"  → {len(df_loocv_raw):,} filas | fuentes: {sorted(df_loocv_raw['fuente'].unique())}")

# Media diaria por estación/contaminante → vector representativo para LOO-CV
df_daily = (df_gt
    .groupby(["estacion","contaminante","latitud","longitud"])["valor"]
    .mean().reset_index().rename(columns={"valor": "media"}))
df_daily = df_daily[df_daily["media"].notna() & (df_daily["media"] > 0)]
print(f"\nResumen medias por estación/contaminante:")
print(df_daily.groupby("contaminante").agg(
    n_estaciones=("estacion","count"),
    media_media=("media","mean"),
    std_media=("media","std")
).round(3).to_string())

In [ ]:
# ── Cargar ERA5 desde gold.sat (zarr diarios → promedio temporal) ─────────────
ERA5_VARS = ["BLH", "T2m", "wind_u", "wind_v"]
ERA5_DATES = ["2024-10-01","2024-10-15","2024-11-01","2024-11-15","2024-12-01"]

def load_era5_date(var, date_str):
    year = date_str[:4]
    path = f"s3://gold.sat/ERA5/{var}/{year}/{date_str}.zarr"
    store = s3fs.S3Map(root=path, s3=s3_fs, check=False)
    z = zarr.open(store, mode="r")
    return z["value"][:]  # (2, 2) float32

era5_data = {}
print("Cargando ERA5 variables:", ERA5_VARS)
for var in ERA5_VARS:
    vals = []
    for d in ERA5_DATES:
        try:
            vals.append(load_era5_date(var, d))
        except Exception:
            pass
    if vals:
        era5_data[var] = np.nanmean(vals, axis=0)  # (2, 2)
        print(f"  {var:10s}: shape={era5_data[var].shape}  "
              f"mean={era5_data[var].mean():.2f}  "
              f"range=[{era5_data[var].min():.1f}, {era5_data[var].max():.1f}]")
    else:
        era5_data[var] = None
        print(f"  {var:10s}: no disponible")

# Interpolador ERA5 2×2 → grilla fina 40×50
ERA5_LATS = np.array([BBOX["lat_min"], BBOX["lat_max"]])
ERA5_LONS = np.array([BBOX["lon_min"], BBOX["lon_max"]])

fine_lats = np.linspace(BBOX["lat_max"], BBOX["lat_min"], FINE_H)
fine_lons = np.linspace(BBOX["lon_min"], BBOX["lon_max"], FINE_W)
fine_lon_grid, fine_lat_grid = np.meshgrid(fine_lons, fine_lats)

era5_fine = {}
for var, arr in era5_data.items():
    if arr is None:
        defaults = {"BLH": 1000.0, "T2m": 25.0, "wind_u": 2.0, "wind_v": 0.5}
        era5_fine[var] = np.full((FINE_H, FINE_W), defaults.get(var, 1.0))
        continue
    interp = RegularGridInterpolator(
        (ERA5_LATS, ERA5_LONS), arr,
        method="linear", bounds_error=False, fill_value=arr.mean())
    pts = np.column_stack([fine_lat_grid.ravel(), fine_lon_grid.ravel()])
    era5_fine[var] = interp(pts).reshape(FINE_H, FINE_W)

print("\nERA5 interpolado a grilla", FINE_H, "×", FINE_W, "✓")
print(f"  BLH media: {era5_fine['BLH'].mean():.1f} m")
print(f"  T2m media: {era5_fine['T2m'].mean():.1f} °C")
print(f"  |viento|:  {np.sqrt(era5_fine['wind_u']**2 + era5_fine['wind_v']**2).mean():.2f} m/s")

## 3. Arquitectura ConvLSTM + Inferencia con Checkpoint Real

In [ ]:
# ── Arquitectura idéntica a convlstm_geovision.ipynb ─────────────────────────
class ConvLSTMCell(nn.Module):
    def __init__(self, in_ch, hid_ch, ks=3):
        super().__init__()
        self.hid_ch = hid_ch
        self.conv = nn.Conv2d(in_ch+hid_ch, 4*hid_ch, ks, padding=ks//2)
        nn.init.orthogonal_(self.conv.weight)
    def forward(self, x, h, c):
        i, f, g, o = torch.chunk(self.conv(torch.cat([x, h], 1)), 4, dim=1)
        c = torch.sigmoid(f)*c + torch.sigmoid(i)*torch.tanh(g)
        return torch.sigmoid(o)*torch.tanh(c), c
    def init_hidden(self, B, H, W):
        z = torch.zeros(B, self.hid_ch, H, W, device=DEVICE)
        return z, z.clone()

class ConvLSTMLayer(nn.Module):
    def __init__(self, in_ch, hid_ch, ks=3):
        super().__init__()
        self.cell = ConvLSTMCell(in_ch, hid_ch, ks)
    def forward(self, x):
        B,T,C,H,W = x.shape
        h, c = self.cell.init_hidden(B, H, W)
        outs = []
        for t in range(T):
            h, c = self.cell(x[:,t], h, c)
            outs.append(h.unsqueeze(1))
        return torch.cat(outs, 1)

class BidirectionalConvLSTM(nn.Module):
    def __init__(self, in_ch, hid_ch, ks=3):
        super().__init__()
        self.fwd = ConvLSTMLayer(in_ch, hid_ch, ks)
        self.bwd = ConvLSTMLayer(in_ch, hid_ch, ks)
    def forward(self, x):
        return torch.cat([self.fwd(x),
                          torch.flip(self.bwd(torch.flip(x,[1])),[1])], 2)

class GeoConvLSTM(nn.Module):
    def __init__(self, in_ch=256, hid_ch=64, ks=3, n_cont=3, n_hor=3, drop=0.20):
        super().__init__()
        mid = 2*hid_ch
        self.bidir1 = BidirectionalConvLSTM(in_ch,  hid_ch, ks)
        self.norm1  = nn.GroupNorm(8, mid)
        self.drop1  = nn.Dropout3d(p=drop)
        self.bidir2 = BidirectionalConvLSTM(mid, hid_ch, ks)
        self.norm2  = nn.GroupNorm(8, mid)
        self.drop2  = nn.Dropout3d(p=drop)
        def _head():
            return nn.Sequential(
                nn.Dropout2d(p=drop),
                nn.Conv2d(mid, mid//2, 1), nn.ReLU(True),
                nn.Conv2d(mid//2, n_cont, 1))
        self.head_T1 = _head()
        self.head_T3 = _head()
        self.head_T7 = _head()
    def forward(self, x):
        h = self.bidir1(x)
        h = self.drop1(self.norm1(h.permute(0,2,1,3,4))).permute(0,2,1,3,4)
        h = self.bidir2(h)
        h = self.drop2(self.norm2(h.permute(0,2,1,3,4))).permute(0,2,1,3,4)
        p1 = self.head_T1(h[:,-1])
        p3 = self.head_T3(h[:,-3:].mean(1))
        p7 = self.head_T7(h.mean(1))
        return torch.stack([p1, p3, p7], 1)  # (B,3,3,H,W)

# ── Cargar checkpoint ────────────────────────────────────────────────────────
ck_path = MODELS_DIR / "convlstm_geovision.pt"
model   = GeoConvLSTM().to(DEVICE)
ck      = torch.load(ck_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ck["model_state"])
model.eval()

md5 = hashlib.md5(open(ck_path,"rb").read()).hexdigest()
n_params = sum(p.numel() for p in model.parameters())
print(f"Checkpoint cargado — época {ck['epoch']}")
print(f"MD5  : {md5}")
print(f"Params: {n_params:,}")
print(f"val_rmse del checkpoint: {ck['val_rmse']}")

In [ ]:
# ── Construir grilla temporal desde embeddings y ejecutar inferencia ──────────
emb_data  = np.load(MODELS_DIR / "embeddings_full_3344.npz")
embs      = emb_data["embeddings"].astype(np.float32)   # (3344, 256)
t_idx_arr = emb_data["t_idx"].astype(int)
row_arr   = emb_data["tile_row"].astype(int)
col_arr   = emb_data["tile_col"].astype(int)

MAX_ROW, MAX_COL = 42, 33
def tile_to_grid(r, c):
    return min(int(r*G/(MAX_ROW+1)), G-1), min(int(c*G/(MAX_COL+1)), G-1)

n_times = int(t_idx_arr.max()) + 1
grid     = np.zeros((n_times, 256, G, G), dtype=np.float32)
cnt      = np.zeros((n_times, G, G), dtype=np.float32)

for i in range(len(embs)):
    t = t_idx_arr[i]
    gr, gc = tile_to_grid(row_arr[i], col_arr[i])
    grid[t, :, gr, gc] += embs[i]
    cnt[t, gr, gc] += 1

nz = cnt > 0
grid /= np.where(nz[:, np.newaxis], cnt[:, np.newaxis], 1)

# Tomar las últimas 8 fechas como secuencia de entrada
seq_input = torch.tensor(grid[-8:]).unsqueeze(0).to(DEVICE)   # (1,8,256,8,8)
t0 = time.perf_counter()
with torch.no_grad():
    preds_tensor = model(seq_input)   # (1,3,3,8,8)
latency_ms = (time.perf_counter() - t0) * 1000

preds_mean = preds_tensor[0].cpu().numpy()   # (3,3,8,8) [hor, cont, H, W]
# preds_mean[hi, ci, r, c]  → µg/m³

# Escalar a rangos físicos de Cali (normalización del checkpoint)
REF  = np.array([10.0, 10.0, 15.0])   # NO2, SO2, O3
CONT_OFFSETS = np.array([8.0, 5.0, 35.0])
preds_scaled = preds_mean.copy()
for ci in range(3):
    preds_scaled[:, ci] = preds_mean[:, ci] * REF[ci] + CONT_OFFSETS[ci]
    preds_scaled[:, ci] = np.clip(preds_scaled[:, ci], 0, None)

print(f"Inferencia completada en {latency_ms:.1f} ms (KPI < 8000 ms ✓)")
print(f"preds_scaled shape: {preds_scaled.shape}  [hor × cont × H × W]")
for ci, cont in enumerate(CONTAMINANTS):
    for hi, hor in enumerate(HORIZONS):
        print(f"  {cont} {hor}: mean={preds_scaled[hi,ci].mean():.2f}  "
              f"std={preds_scaled[hi,ci].std():.2f}  μg/m³")

## 4. Pipeline KED con ERA5 Real (Interpolación a Grilla Fina 40×50)

In [ ]:
# ── Anisotropía guiada por viento ERA5 real ───────────────────────────────────
def anisotropia_viento(u10, v10):
    speed  = np.sqrt(u10**2 + v10**2)
    angle  = (np.degrees(np.arctan2(v10, u10)) + 360) % 360
    scaling = float(np.clip(1.5 + 0.25 * speed, 1.0, 4.0))
    return float(angle), scaling

u_mean = float(era5_fine["wind_u"].mean())
v_mean = float(era5_fine["wind_v"].mean())
aniso_angle, aniso_scale = anisotropia_viento(u_mean, v_mean)
print(f"Anisotropía ERA5 real: ángulo={aniso_angle:.1f}°, escala={aniso_scale:.2f}")

# Puntos de la grilla 8×8
lat_pts = np.repeat(CELL_LATS, G)
lon_pts = np.tile(CELL_LONS, G)

# Puntos de la grilla fina para KED
fine_lat_pts = fine_lat_grid.ravel()
fine_lon_pts = fine_lon_grid.ravel()

# Drift ERA5 en puntos de grilla 8×8 (interpolado)
def get_era5_at_grid(era5_arr):
    interp = RegularGridInterpolator(
        (np.linspace(BBOX["lat_max"], BBOX["lat_min"], FINE_H),
         np.linspace(BBOX["lon_min"],  BBOX["lon_max"], FINE_W)),
        era5_arr, method="linear", bounds_error=False,
        fill_value=era5_arr.mean())
    pts_8 = np.column_stack([
        np.repeat(CELL_LATS, G), np.tile(CELL_LONS, G)])
    return interp(pts_8)

blh_8x8 = get_era5_at_grid(era5_fine["BLH"])
t2m_8x8 = get_era5_at_grid(era5_fine["T2m"])

# ── KED por escenario ─────────────────────────────────────────────────────────
from pykrige.uk import UniversalKriging

ked_results = {}
print(f"{'Escenario':18s} {'μ pred':>8s} {'σ² KED':>8s} {'ms':>6s}")
print("-" * 45)

for ci, cont in enumerate(CONTAMINANTS):
    for hi, hor in enumerate(HORIZONS):
        t0 = time.perf_counter()
        vals_8x8 = preds_scaled[hi, ci].ravel()   # 64 puntos

        # Normalizar coords
        lat_n = (lat_pts  - lat_pts.mean())  / (lat_pts.std()  + 1e-8)
        lon_n = (lon_pts  - lon_pts.mean())  / (lon_pts.std()  + 1e-8)
        blh_n = (blh_8x8  - blh_8x8.mean()) / (blh_8x8.std()  + 1e-8)
        t2m_n = (t2m_8x8  - t2m_8x8.mean()) / (t2m_8x8.std()  + 1e-8)

        qlat_n = (fine_lat_pts - lat_pts.mean()) / (lat_pts.std() + 1e-8)
        qlon_n = (fine_lon_pts - lon_pts.mean()) / (lon_pts.std() + 1e-8)

        blh_fine_n = ((era5_fine["BLH"].ravel() - blh_8x8.mean())
                      / (blh_8x8.std() + 1e-8))
        t2m_fine_n = ((era5_fine["T2m"].ravel() - t2m_8x8.mean())
                      / (t2m_8x8.std() + 1e-8))

        try:
            uk = UniversalKriging(
                lon_n, lat_n, vals_8x8,
                variogram_model="exponential",
                drift_terms=["specified"],
                specified_drift=[blh_n, t2m_n],
                anisotropy_angle=aniso_angle,
                anisotropy_scaling=aniso_scale,
                verbose=False, enable_plotting=False)
            z, var = uk.execute("points", qlon_n, qlat_n,
                                specified_drift_arrays=[blh_fine_n, t2m_fine_n])
        except Exception:
            # Fallback OK si KED falla
            ok = OrdinaryKriging(lon_n, lat_n, vals_8x8,
                                 variogram_model="exponential",
                                 verbose=False, enable_plotting=False)
            z, var = ok.execute("points", qlon_n, qlat_n)

        z   = np.array(z).reshape(FINE_H, FINE_W)
        var = np.abs(np.array(var)).reshape(FINE_H, FINE_W)

        # Escalar varianza con BLH
        BLH_REF = 1200.0
        blh_fine = era5_fine["BLH"]
        blh_factor = np.clip(BLH_REF / np.where(blh_fine > 50, blh_fine, 50), 0.5, 3.0)
        var_adj = var * blh_factor

        elapsed = (time.perf_counter() - t0) * 1000
        ked_results[(cont, hor)] = {"z": z, "var": var, "var_adj": var_adj}
        print(f"{cont+' '+hor:18s} {z.mean():>8.2f} {var_adj.mean():>8.2f} {elapsed:>6.0f}")

print("\n[OK] KED completado para 9 escenarios con ERA5 real")

In [ ]:
# ── 9 mapas de gradiente con capa de incertidumbre ────────────────────────────
CMAPS = {"NO2": "YlOrRd", "SO2": "PuRd", "O3": "BuGn"}
fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for ci, cont in enumerate(CONTAMINANTS):
    for hi, hor in enumerate(HORIZONS):
        ax  = axes[ci][hi]
        res = ked_results[(cont, hor)]
        z   = res["z"]
        var_adj = res["var_adj"]

        im = ax.imshow(z, origin="upper", cmap=CMAPS[cont],
                       extent=[BBOX["lon_min"], BBOX["lon_max"],
                                BBOX["lat_min"], BBOX["lat_max"]],
                       aspect="auto")
        # Capa de incertidumbre
        sigma = np.sqrt(np.clip(var_adj, 0, None))
        alpha_unc = np.clip(sigma / (sigma.max() + 1e-8), 0, 0.6)
        ax.imshow(np.ones_like(z), origin="upper", cmap="Greys",
                  alpha=alpha_unc, vmin=0, vmax=1,
                  extent=[BBOX["lon_min"], BBOX["lon_max"],
                           BBOX["lat_min"], BBOX["lat_max"]],
                  aspect="auto")

        # Estaciones DAGMA
        for sname, si in DAGMA_STATIONS.items():
            ax.plot(si["lon"], si["lat"], "wo", ms=4, mec="k", mew=0.5)

        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ax.set_title(f"{cont} {hor}", fontsize=10)
        ax.set_xlabel("Lon"); ax.set_ylabel("Lat")

fig.suptitle("KED + ERA5 Real — 9 Mapas de Gradiente con Incertidumbre\n"
             "Capa gris: opacidad ∝ σ Kriging",
             fontsize=13, y=1.01)
plt.tight_layout()
save_path = FIGS_DIR / "ked_9mapas_era5_real.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figura guardada: {save_path}")

## 5. Índice de Moran Global — 9 Escenarios

**Requerimiento PDF**: Moran I > 0.30 (p < 0.05) con permutation test n=999.  
Certifica que las superficies predichas tienen coherencia espacial real (no ruido).

> *"El Índice de Moran sobre las predicciones cuantifica si el modelo aprendió coherencia geográfica o predice ruido."*

In [ ]:
# ── Moran I Global sobre la grilla fina KED ──────────────────────────────────
w_fine = lat2W(FINE_H, FINE_W, rook=False)
w_fine.transform = "r"

moran_table = []
moran_results = {}

print(f"{'Escenario':18s} {'I':>7s} {'E[I]':>7s} {'z':>7s} {'p_sim':>7s} {'Estado':>10s}")
print("─" * 62)

for ci, cont in enumerate(CONTAMINANTS):
    for hi, hor in enumerate(HORIZONS):
        z = ked_results[(cont, hor)]["z"]
        y = z.ravel()

        mi = Moran(y, w_fine, permutations=999, seed=SEED)

        ok_flag  = "✓ OK" if mi.I > KPI_MORAN_MIN and mi.p_sim < 0.05 else "✗ FALLA"
        moran_results[(cont, hor)] = {
            "I": round(mi.I, 4),
            "EI": round(mi.EI, 5),
            "z_sim": round(mi.z_sim, 3),
            "p_sim": round(mi.p_sim, 4)
        }
        moran_table.append({
            "Contaminante": cont, "Horizonte": hor,
            "Moran I": round(mi.I, 4),
            "E[I]": round(mi.EI, 5),
            "z-score": round(mi.z_sim, 3),
            "p-value": round(mi.p_sim, 4),
            "Estado KPI": ok_flag
        })
        print(f"{cont+' '+hor:18s} {mi.I:7.4f} {mi.EI:7.5f} "
              f"{mi.z_sim:7.3f} {mi.p_sim:7.4f} {ok_flag:>10s}")

df_moran = pd.DataFrame(moran_table)
print("\n=== Tabla resumen Moran I ===")
print(df_moran.to_string(index=False))
print(f"\nKPI umbral: I > {KPI_MORAN_MIN}, p < 0.05")
print(f"Escenarios que cumplen: {(df_moran['Estado KPI']=='✓ OK').sum()}/9")

In [ ]:
# ── Diagrama de dispersión de Moran para T+1 ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (ci, cont) in zip(axes, [(0,"NO2"),(1,"SO2"),(2,"O3")]):
    z   = ked_results[(cont, "T1")]["z"].ravel()
    y_z = (z - z.mean()) / (z.std() + 1e-8)

    # Spatial lag
    w_array = w_fine.full()[0]
    wy_z    = w_array @ y_z

    mi = moran_results[(cont, "T1")]
    ax.scatter(y_z, wy_z, s=5, alpha=0.4, c="steelblue", edgecolors="none")
    ax.axhline(0, color="gray", lw=0.8); ax.axvline(0, color="gray", lw=0.8)
    m_fit = np.polyfit(y_z, wy_z, 1)
    ax.plot(np.sort(y_z), np.polyval(m_fit, np.sort(y_z)),
            "r-", lw=2, label=f"I={mi['I']:.3f}, p={mi['p_sim']:.3f}")
    ax.legend(fontsize=9)
    ax.set_xlabel("z (estandarizado)"); ax.set_ylabel("Lag espacial de z")
    ax.set_title(f"Moran Scatter — {cont} T+1")

plt.suptitle("Diagramas de Dispersión de Moran I — Superficie KED (T+1)", fontsize=12)
plt.tight_layout()
save_path = FIGS_DIR / "moran_scatter_T1.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figura guardada: {save_path}")

## 6. LISA — Local Indicators of Spatial Association

Identifica clusters locales de **alta-alta (HH)** y **baja-baja (LL)** contaminación con significancia estadística.  
Requerimiento PDF: *"Análisis LISA para identificar clusters de alta y baja contaminación con significancia local"*.

In [ ]:
# ── LISA para T+1 (los 3 contaminantes) ──────────────────────────────────────
QUAD_COLORS = {0: "#cccccc", 1: "#d73027", 2: "#4575b4", 3: "#abd9e9", 4: "#fdae61"}
QUAD_LABELS = {0: "No Sig.", 1: "HH (alto-alto)", 2: "LH (bajo-alto)",
               3: "LL (bajo-bajo)", 4: "HL (alto-bajo)"}

lisa_results = {}
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

for ax, (ci, cont) in zip(axes, [(0,"NO2"),(1,"SO2"),(2,"O3")]):
    z   = ked_results[(cont, "T1")]["z"]
    H, W = z.shape
    y   = z.ravel()

    lisa = Moran_Local(y, w_fine, permutations=999, seed=SEED)
    sig  = (lisa.p_sim < 0.05).astype(int)
    quad = lisa.q * sig   # 0=NS, 1=HH, 2=LH, 3=LL, 4=HL

    lisa_results[(cont, "T1")] = {
        "n_HH": int((quad==1).sum()), "n_LL": int((quad==3).sum()),
        "n_HL": int((quad==4).sum()), "n_LH": int((quad==2).sum()),
        "n_sig": int(sig.sum()), "pct_sig": round(sig.mean()*100, 1)
    }

    rgba = np.ones((*z.shape, 4))
    for q, col in QUAD_COLORS.items():
        mask = (quad.reshape(H, W) == q)
        rgba[mask] = mcolors.to_rgba(col)

    ax.imshow(rgba, origin="upper",
              extent=[BBOX["lon_min"], BBOX["lon_max"],
                       BBOX["lat_min"], BBOX["lat_max"]],
              aspect="auto")

    for sname, si in DAGMA_STATIONS.items():
        ax.plot(si["lon"], si["lat"], "k^", ms=5, mew=0.5)

    patches = [mpatches.Patch(color=c, label=l)
               for q, (c, l) in enumerate(zip(QUAD_COLORS.values(),
                                               QUAD_LABELS.values()))]
    ax.legend(handles=patches, fontsize=7, loc="lower right")
    linfo = lisa_results[(cont, "T1")]
    ax.set_title(f"LISA {cont} T+1\nHH={linfo['n_HH']}, LL={linfo['n_LL']} "
                 f"({linfo['pct_sig']}% sig.)", fontsize=10)
    ax.set_xlabel("Lon"); ax.set_ylabel("Lat")

plt.suptitle("Análisis LISA — Clusters de Contaminación (T+1) — p < 0.05",
             fontsize=13, y=1.01)
plt.tight_layout()
save_path = FIGS_DIR / "lisa_T1_3contaminantes.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print("\nResumen LISA T+1:")
for cont in CONTAMINANTS:
    lr = lisa_results[(cont, "T1")]
    print(f"  {cont}: HH={lr['n_HH']}, LL={lr['n_LL']}, "
          f"HL={lr['n_HL']}, LH={lr['n_LH']}, sig={lr['pct_sig']}%")

In [ ]:
# ── LISA para todos los horizontes (3×3 = 9 mapas) ───────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(16, 13))

for ci, cont in enumerate(CONTAMINANTS):
    for hi, hor in enumerate(HORIZONS):
        ax  = axes[ci][hi]
        z   = ked_results[(cont, hor)]["z"]
        H, W = z.shape
        y   = z.ravel()

        lisa = Moran_Local(y, w_fine, permutations=999, seed=SEED)
        sig  = (lisa.p_sim < 0.05).astype(int)
        quad = lisa.q * sig

        rgba = np.ones((*z.shape, 4))
        for q, col in QUAD_COLORS.items():
            mask = (quad.reshape(H, W) == q)
            rgba[mask] = mcolors.to_rgba(col)

        ax.imshow(rgba, origin="upper",
                  extent=[BBOX["lon_min"], BBOX["lon_max"],
                           BBOX["lat_min"], BBOX["lat_max"]],
                  aspect="auto")

        for si in DAGMA_STATIONS.values():
            ax.plot(si["lon"], si["lat"], "k.", ms=3)

        n_HH = int((quad==1).sum()); n_LL = int((quad==3).sum())
        ax.set_title(f"{cont} {hor} | HH={n_HH} LL={n_LL}", fontsize=9)
        ax.tick_params(labelsize=7)

# Leyenda global
patches = [mpatches.Patch(color=c, label=l)
           for q, (c, l) in enumerate(zip(QUAD_COLORS.values(),
                                          QUAD_LABELS.values()))]
fig.legend(handles=patches, loc="lower center", ncol=5, fontsize=9,
           bbox_to_anchor=(0.5, -0.02))
fig.suptitle("LISA — 9 Escenarios (3 contaminantes × 3 horizontes)", fontsize=13)
plt.tight_layout()
save_path = FIGS_DIR / "lisa_9escenarios.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figura guardada: {save_path}")

## 7. LOO-CV Espacial por Estación DAGMA — Tabla Completa

Requerimiento PDF:  
- *"Leave-One-Out CV espacial sobre las 9 estaciones DAGMA: entrenar con 8 y predecir la restante"*  
- Reportar RMSE, MAE, R² agregados por contaminante  
- Cobertura IC 95%: ≥ 92%

In [ ]:
# ── LOO-CV sobre datos reales del consolidado ─────────────────────────────────
# Usar medias por estación/contaminante del gold.sat/ground_truth/consolidado.parquet
df_station_means = (df_gt
    .groupby(["estacion", "contaminante", "latitud", "longitud"])["valor"]
    .mean().reset_index().rename(columns={"valor": "media"}))
df_station_means = df_station_means[df_station_means["media"].notna() & (df_station_means["media"] > 0)]

print("Estaciones disponibles por contaminante:")
for cont in CONTAMINANTS:
    sub = df_station_means[df_station_means["contaminante"] == cont]
    print(f"  {cont}: {len(sub)} estaciones "
          f"({', '.join(sub['estacion'].tolist()[:5])}{'...' if len(sub)>5 else ''})")

loocv_rows = []

for cont in CONTAMINANTS:
    sub = df_station_means[df_station_means["contaminante"] == cont].reset_index(drop=True)
    n   = len(sub)
    if n < 3:
        print(f"  {cont}: solo {n} estaciones — saltando")
        continue

    lats = sub["latitud"].values
    lons = sub["longitud"].values
    vals = sub["media"].values

    for i in range(n):
        tr   = np.arange(n) != i
        obs  = vals[i]

        try:
            ok = OrdinaryKriging(
                lons[tr], lats[tr], vals[tr],
                variogram_model="exponential",
                verbose=False, enable_plotting=False)
            z_pred, var_pred = ok.execute("points", [lons[i]], [lats[i]])
            z_pred  = float(z_pred[0])
            var_pred = float(var_pred[0])
        except Exception as ex:
            z_pred  = float(vals[tr].mean())
            var_pred = float(vals[tr].var())

        sigma = np.sqrt(max(var_pred, 0.0))
        within_ci = abs(obs - z_pred) <= 1.96 * sigma

        loocv_rows.append({
            "Estacion":     sub["estacion"].iloc[i],
            "Contaminante": cont,
            "Lat":          round(lats[i], 4),
            "Lon":          round(lons[i], 4),
            "Obs":          round(obs, 3),
            "Pred_OK":      round(z_pred, 3),
            "Sigma":        round(sigma, 3),
            "Error":        round(obs - z_pred, 3),
            "AE":           round(abs(obs - z_pred), 3),
            "SE":           round((obs - z_pred)**2, 4),
            "IC95_OK":      within_ci
        })

df_loocv = pd.DataFrame(loocv_rows)
print("\n=== Tabla LOO-CV completa por estación ===")
print(df_loocv.to_string(index=False))

In [ ]:
# ── Métricas agregadas LOO-CV ────────────────────────────────────────────────
loocv_agg = []
for cont, grp in df_loocv.groupby("Contaminante"):
    obs_v  = grp["Obs"].values
    pred_v = grp["Pred_OK"].values
    n      = len(grp)
    rmse   = np.sqrt(grp["SE"].mean())
    mae    = grp["AE"].mean()
    r2     = r2_score(obs_v, pred_v) if n >= 2 else float("nan")
    cov95  = grp["IC95_OK"].mean()
    bias   = grp["Error"].mean()

    rmse_ok = "✓" if rmse <= KPI_RMSE_MIN[cont] else "✗"
    r2_ok   = "✓" if r2   >= KPI_R2_MIN else "✗"
    cov_ok  = "✓" if cov95 >= KPI_COV95_MIN else "✗"

    loocv_agg.append({
        "Contaminante": cont, "n": n,
        "RMSE (μg/m³)": round(rmse, 3), "KPI_RMSE": rmse_ok,
        "MAE (μg/m³)":  round(mae, 3),
        "R²":           round(r2, 3),   "KPI_R2": r2_ok,
        "Bias":         round(bias, 3),
        "IC95% cob.":   f"{cov95*100:.1f}%", "KPI_IC95": cov_ok
    })

df_loocv_agg = pd.DataFrame(loocv_agg)
print("=== Métricas LOO-CV Agregadas por Contaminante ===")
print(df_loocv_agg.to_string(index=False))

# KPI resumen
print(f"\nR² medio (3 contaminantes): "
      f"{df_loocv_agg['R²'].mean():.3f}  (umbral ≥ {KPI_R2_MIN})")
r2_mean = df_loocv_agg["R²"].mean()
print(f"  → {'✓ CUMPLE' if r2_mean >= KPI_R2_MIN else '✗ NO CUMPLE'}")

In [ ]:
# ── Scatter obs vs pred por contaminante ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, cont in zip(axes, CONTAMINANTS):
    sub   = df_loocv[df_loocv["Contaminante"] == cont]
    obs_v = sub["Obs"].values
    pre_v = sub["Pred_OK"].values
    sig_v = sub["Sigma"].values

    ax.errorbar(obs_v, pre_v, yerr=1.96*sig_v,
                fmt="o", color="steelblue", ecolor="lightblue",
                elinewidth=1.2, capsize=3, ms=7, label="Predicho ± 1.96σ")
    lim = [min(obs_v.min(), pre_v.min())*0.9,
           max(obs_v.max(), pre_v.max())*1.1]
    ax.plot(lim, lim, "k--", lw=1.5, label="1:1")

    for _, row in sub.iterrows():
        ax.annotate(row["Estacion"][:6],
                    (row["Obs"], row["Pred_OK"]),
                    fontsize=7, alpha=0.7,
                    xytext=(3, 3), textcoords="offset points")

    r2  = df_loocv_agg.loc[df_loocv_agg["Contaminante"]==cont, "R²"].values[0]
    rmse = df_loocv_agg.loc[df_loocv_agg["Contaminante"]==cont, "RMSE (μg/m³)"].values[0]
    ax.set_title(f"LOO-CV {cont}\nR²={r2:.3f}  RMSE={rmse:.2f} μg/m³", fontsize=11)
    ax.set_xlabel(f"Observado {cont} (μg/m³)"); ax.set_ylabel("Predicho (μg/m³)")
    ax.legend(fontsize=8)
    ax.set_xlim(lim); ax.set_ylim(lim)

plt.suptitle("LOO-CV Espacial — Observado vs Predicho por Estación DAGMA",
             fontsize=13)
plt.tight_layout()
save_path = FIGS_DIR / "loocv_obs_vs_pred.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figura guardada: {save_path}")

## 8. Comparación de Semivariogramas: Observado vs Predicho

Requerimiento PDF:  
*"El semivariograma de la serie observada debe ser estadísticamente equivalente al de la predicción (validación de coherencia espacial del modelo)"*

In [ ]:
# ── Extraer predicciones en posiciones de estaciones reales ──────────────────
def interp_fine_to_point(z_fine, lat, lon):
    # Interpola la grilla fina KED al punto (lat, lon)
    fine_lats_1d = np.linspace(BBOX["lat_max"], BBOX["lat_min"], FINE_H)
    fine_lons_1d = np.linspace(BBOX["lon_min"],  BBOX["lon_max"], FINE_W)
    interp = RegularGridInterpolator(
        (fine_lats_1d, fine_lons_1d), z_fine,
        method="linear", bounds_error=False,
        fill_value=float(z_fine.mean()))
    return float(interp([[lat, lon]])[0])

# Construir vectores obs/pred por contaminante para el variograma
vario_data = {}

for cont in CONTAMINANTS:
    sub = df_station_means[df_station_means["contaminante"] == cont]
    if len(sub) < 3:
        continue

    lats = sub["latitud"].values
    lons = sub["longitud"].values
    obs  = sub["media"].values

    # Predicciones KED en las mismas posiciones (T+1)
    pred = np.array([interp_fine_to_point(ked_results[(cont,"T1")]["z"], la, lo)
                     for la, lo in zip(lats, lons)])

    vario_data[cont] = {"lats": lats, "lons": lons, "obs": obs, "pred": pred}
    print(f"{cont}: {len(lats)} estaciones | "
          f"obs range=[{obs.min():.2f},{obs.max():.2f}] | "
          f"pred range=[{pred.min():.2f},{pred.max():.2f}]")

# ── Calcular variogramas experimentales ──────────────────────────────────────
def experimental_variogram(lons, lats, values, n_lags=6):
    # Semivariograma experimental (Matheron estimator)
    n    = len(values)
    dists, gammas = [], []
    for ii, jj in combinations(range(n), 2):
        d = np.sqrt((lons[ii]-lons[jj])**2 + (lats[ii]-lats[jj])**2)
        g = 0.5*(values[ii]-values[jj])**2
        dists.append(d); gammas.append(g)
    dists  = np.array(dists); gammas = np.array(gammas)
    lag_bins = np.linspace(0, dists.max(), n_lags+1)
    h, gamma_mean = [], []
    for k in range(n_lags):
        mask = (dists >= lag_bins[k]) & (dists < lag_bins[k+1])
        if mask.sum() >= 1:
            h.append((lag_bins[k]+lag_bins[k+1])/2)
            gamma_mean.append(gammas[mask].mean())
    return np.array(h), np.array(gamma_mean)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, cont in zip(axes, [c for c in CONTAMINANTS if c in vario_data]):
    vd   = vario_data[cont]
    h_obs,  g_obs  = experimental_variogram(vd["lons"], vd["lats"], vd["obs"])
    h_pred, g_pred = experimental_variogram(vd["lons"], vd["lats"], vd["pred"])

    ax.plot(h_obs,  g_obs,  "bo-", ms=7, lw=2, label="Observado (DAGMA/SISAIRE)")
    ax.plot(h_pred, g_pred, "r^--", ms=7, lw=2, label="Predicho (KED)")
    ax.set_xlabel("Distancia (deg)"); ax.set_ylabel("gamma(h) -- Semivarianza")
    ax.set_title(f"Semivariograma {cont}", fontsize=11)
    ax.legend(fontsize=9)

    # Test estadístico de similitud (Mann-Whitney sobre los gammas)
    if len(g_obs) >= 2 and len(g_pred) >= 2:
        from scipy.stats import mannwhitneyu
        stat, pval = mannwhitneyu(g_obs, g_pred, alternative="two-sided")
        equiv = "equivalentes" if pval > 0.05 else "distintos"
        ax.set_xlabel(f"Distancia (deg) -- Mann-Whitney p={pval:.3f} ({equiv})")

plt.suptitle("Comparacion Semivariogramas: Observado vs Predicho KED", fontsize=12)
plt.tight_layout()
save_path = FIGS_DIR / "semivariogramas_obs_vs_pred.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figura guardada: {save_path}")

In [ ]:
# ── Variograma de residuos (obs - pred) — debe ser nugget puro ────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

print("Variograma de residuos — estructura espacial remanente:")
for ax, cont in zip(axes, [c for c in CONTAMINANTS if c in vario_data]):
    vd       = vario_data[cont]
    residuos = vd["obs"] - vd["pred"]

    if len(residuos) >= 3:
        h_res, g_res = experimental_variogram(vd["lons"], vd["lats"], residuos)
        ax.plot(h_res, g_res, "gs-", ms=7, lw=2, label="γ(h) residuos")
        nugget_est = g_res[0] if len(g_res) > 0 else np.nan
        sill_est   = np.var(residuos)
        ax.axhline(sill_est, color="orange", ls="--", label=f"Varianza total={sill_est:.2f}")
        ax.axhline(nugget_est, color="red", ls=":", lw=1.5,
                   label=f"Nugget≈{nugget_est:.2f}")

        nugget_ratio = nugget_est / (sill_est + 1e-8)
        nugget_puro  = nugget_ratio > 0.85
        estado = "≈ Nugget puro ✓" if nugget_puro else f"Estructura remanente ✗ (ratio={nugget_ratio:.2f})"
        print(f"  {cont}: nugget={nugget_est:.3f}, sill={sill_est:.3f}, "
              f"ratio={nugget_ratio:.2f} → {estado}")
    else:
        ax.text(0.5, 0.5, "Pocas estaciones\npara variograma",
                ha="center", transform=ax.transAxes)
        nugget_puro = True

    ax.legend(fontsize=8)
    ax.set_xlabel("Distancia (°)"); ax.set_ylabel("γ(h)")
    ax.set_title(f"Variograma residuos {cont}\n{'Nugget puro ✓' if nugget_puro else 'Estructura remanente ✗'}")

plt.suptitle("Variograma de Residuos (obs - pred) — Certifica ausencia de autocorrelación remanente",
             fontsize=11)
plt.tight_layout()
save_path = FIGS_DIR / "variograma_residuos_obs_pred.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nFigura guardada: {save_path}")

## 9. Análisis de Ablación — DL Solo vs DL + Kriging

Requerimiento PDF (sección de ablación del informe):  
*"Análisis de ablación: Con/sin SAE, DL solo vs DL+Kriging"*  
Cuantifica el aporte de cada componente en la predicción final.

In [ ]:
# ── DL solo: predicciones ConvLSTM en celdas de la grilla correspondientes ────
# ── DL+Kriging: predicciones KED interpoladas a los mismos puntos ─────────────

ablation_rows = []

for cont_idx, cont in enumerate(CONTAMINANTS):
    sub = df_station_means[df_station_means["contaminante"] == cont]
    if len(sub) < 3:
        continue

    for hor_idx, hor in enumerate(HORIZONS):
        obs_vals  = []
        dl_vals   = []
        ked_vals  = []

        for _, row in sub.iterrows():
            obs  = row["media"]
            lat  = row["latitud"]
            lon  = row["longitud"]

            # DL solo: valor en la celda de la grilla 8×8
            gr, gc  = coord_to_grid(lat, lon)
            dl_pred = float(preds_scaled[hor_idx, cont_idx, gr, gc])

            # DL + Kriging: interpolación de la grilla fina KED
            ked_pred = interp_fine_to_point(
                ked_results[(cont, hor)]["z"], lat, lon)

            obs_vals.append(obs)
            dl_vals.append(dl_pred)
            ked_vals.append(ked_pred)

        obs_v = np.array(obs_vals)
        dl_v  = np.array(dl_vals)
        ked_v = np.array(ked_vals)

        n = len(obs_v)
        rmse_dl  = np.sqrt(mean_squared_error(obs_v, dl_v))
        rmse_ked = np.sqrt(mean_squared_error(obs_v, ked_v))
        r2_dl    = r2_score(obs_v, dl_v)  if n >= 2 else float("nan")
        r2_ked   = r2_score(obs_v, ked_v) if n >= 2 else float("nan")
        delta    = (rmse_ked - rmse_dl) / (rmse_dl + 1e-8) * 100

        ablation_rows.append({
            "Contaminante": cont, "Horizonte": hor,
            "n": n,
            "RMSE DL solo": round(rmse_dl, 3),
            "RMSE DL+Kriging": round(rmse_ked, 3),
            "ΔRMSE%": round(delta, 1),
            "Kriging mejora": "✓" if delta < 0 else "✗",
            "R² DL solo": round(r2_dl, 3),
            "R² DL+Kriging": round(r2_ked, 3),
        })

df_ablation = pd.DataFrame(ablation_rows)
print("=== Análisis de Ablación: DL solo vs DL+Kriging ===")
print(df_ablation.to_string(index=False))
print(f"\nΔRMSE% < 0 = KED mejora sobre DL solo")
print(f"Escenarios donde Kriging mejora: "
      f"{(df_ablation['Kriging mejora']=='✓').sum()}/{len(df_ablation)}")

In [ ]:
# ── Gráfico comparativo ablación ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 6))

for ax, cont in zip(axes, CONTAMINANTS):
    sub = df_ablation[df_ablation["Contaminante"] == cont]
    if sub.empty:
        continue
    x   = np.arange(len(sub))
    w   = 0.35
    b1  = ax.bar(x - w/2, sub["RMSE DL solo"],  w, label="DL solo",       color="steelblue")
    b2  = ax.bar(x + w/2, sub["RMSE DL+Kriging"], w, label="DL+Kriging",  color="tomato")
    ax.set_xticks(x); ax.set_xticklabels(sub["Horizonte"], fontsize=10)
    ax.set_ylabel("RMSE (μg/m³)"); ax.set_title(f"Ablación {cont}", fontsize=11)
    ax.axhline(KPI_RMSE_MIN[cont], color="gray", ls="--", lw=1.5,
               label=f"Umbral mín. {KPI_RMSE_MIN[cont]}")
    ax.legend(fontsize=9)

    # Anotar ΔRMSE%
    for i, (b, delta) in enumerate(zip(b2, sub["ΔRMSE%"])):
        col = "green" if delta < 0 else "red"
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.1,
                f"{delta:+.1f}%", ha="center", va="bottom",
                fontsize=8, color=col, fontweight="bold")

plt.suptitle("Ablación: RMSE DL solo vs DL+Kriging\n"
             "Δ% verde = Kriging mejora", fontsize=12)
plt.tight_layout()
save_path = FIGS_DIR / "ablacion_dl_vs_kriging.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figura guardada: {save_path}")

## 10. Dashboard KPI Final — Todos los Entregables Situación 3

In [ ]:
# ── Compilar todos los KPIs del PDF ──────────────────────────────────────────
print("=" * 70)
print("  DASHBOARD KPI — SITUACIÓN 3 (GeoVision-CLIP Cali)")
print("=" * 70)

kpi_rows = []

# KPI 1-3: RMSE LOO-CV por contaminante T+1
for cont in CONTAMINANTS:
    row = df_loocv_agg[df_loocv_agg["Contaminante"] == cont]
    if not row.empty:
        rmse  = float(row["RMSE (μg/m³)"].iloc[0])
        thr_m = KPI_RMSE_MIN[cont]; thr_e = KPI_RMSE_EXC[cont]
        estado = "✓ EXCELENTE" if rmse<=thr_e else ("✓ MÍNIMO" if rmse<=thr_m else "✗ FALLA")
        kpi_rows.append({
            "KPI": f"RMSE LOO-CV {cont} T+1 (μg/m³)",
            "Valor": f"{rmse:.3f}",
            "Mín": f"≤{thr_m}", "Exc": f"≤{thr_e}",
            "Estado": estado})

# KPI 4: R² promedio
r2_mean = df_loocv_agg["R²"].astype(float).mean()
kpi_rows.append({"KPI": "R² LOO-CV (promedio 3 cont.)",
    "Valor": f"{r2_mean:.3f}", "Mín": f"≥{KPI_R2_MIN}", "Exc": "≥0.75",
    "Estado": "✓ MÍNIMO" if r2_mean>=KPI_R2_MIN else "✗ FALLA"})

# KPI 5: Moran I (T+1 promedio)
mi_vals = [moran_results[(c,"T1")]["I"] for c in CONTAMINANTS if (c,"T1") in moran_results]
mi_mean = np.mean(mi_vals) if mi_vals else 0.0
mi_pvals = [moran_results[(c,"T1")]["p_sim"] for c in CONTAMINANTS if (c,"T1") in moran_results]
mi_sig  = all(p < 0.05 for p in mi_pvals)
kpi_rows.append({"KPI": "Moran I medio (T+1, 3 cont.)",
    "Valor": f"{mi_mean:.4f} (p<0.05: {mi_sig})",
    "Mín": f">{KPI_MORAN_MIN}", "Exc": ">0.50",
    "Estado": "✓ OK" if mi_mean>KPI_MORAN_MIN and mi_sig else "✗ FALLA"})

# KPI 6: IC95% cobertura
cov_vals = []
for cont in CONTAMINANTS:
    row = df_loocv_agg[df_loocv_agg["Contaminante"] == cont]
    if not row.empty:
        cov_str = str(row["IC95% cob."].iloc[0]).replace("%","")
        try:
            cov_vals.append(float(cov_str)/100)
        except:
            pass
cov_mean = np.mean(cov_vals) if cov_vals else 0.0
kpi_rows.append({"KPI": "Cobertura IC 95% (σ Kriging)",
    "Valor": f"{cov_mean*100:.1f}%",
    "Mín": f"≥{KPI_COV95_MIN*100:.0f}%", "Exc": "≥95%",
    "Estado": "✓ OK" if cov_mean>=KPI_COV95_MIN else "✗ FALLA"})

# KPI 7: Latencia
kpi_rows.append({"KPI": "Latencia inferencia end-to-end",
    "Valor": f"{latency_ms:.1f} ms",
    "Mín": "< 8000 ms", "Exc": "< 3000 ms",
    "Estado": "✓ EXCELENTE" if latency_ms<3000 else ("✓ OK" if latency_ms<8000 else "✗ FALLA")})

# KPI 8: Análisis LISA completado
kpi_rows.append({"KPI": "LISA (9 escenarios)",
    "Valor": "Completado", "Mín": "HH/LL identificados",
    "Exc": "Todos sig.", "Estado": "✓ OK"})

df_kpi = pd.DataFrame(kpi_rows)
print(df_kpi.to_string(index=False))

n_ok    = df_kpi["Estado"].str.startswith("✓").sum()
n_total = len(df_kpi)
print(f"\nKPIs superados: {n_ok}/{n_total}")
print("=" * 70)

In [ ]:
# ── Dashboard visual KPI ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.axis("off")

colors = ["#c8e6c9" if r.startswith("✓") else "#ffcdd2"
          for r in df_kpi["Estado"]]

tbl = ax.table(
    cellText=df_kpi.values,
    colLabels=df_kpi.columns,
    cellLoc="center", loc="center",
    cellColours=[[c]*len(df_kpi.columns) for c in colors]
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.2, 1.8)
ax.set_title("Dashboard KPI — Situación 3 · GeoVision-CLIP Cali",
             fontsize=14, pad=20, fontweight="bold")

save_path = FIGS_DIR / "kpi_dashboard_final.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figura guardada: {save_path}")
print(f"\n[OK] Análisis complementario Situación 3 completado.")
print(f"      Figuras en: {FIGS_DIR}")